# Week 5 - ML Experiment v1

This notebook creates reproducible evidence for Task/Y, split, baseline comparison, metrics, and Top-3 failure cases.

In [1]:
from pathlib import Path
import sys
import time
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from rule_baseline import predict_batch
df = pd.read_csv(ROOT / 'data' / 'customer_intent_demo.csv')
print('shape:', df.shape)
print(df['intent'].value_counts().sort_index())

shape: (100, 2)
intent
account_login     25
order_delivery    25
product_info      25
refund_return     25
Name: count, dtype: int64


In [2]:
train_df, temp_df = train_test_split(df, test_size=0.30, random_state=42, stratify=df['intent'])
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42, stratify=temp_df['intent'])
train_df = train_df.reset_index(drop=False).rename(columns={'index':'case_id'})
val_df = val_df.reset_index(drop=False).rename(columns={'index':'case_id'})
test_df = test_df.reset_index(drop=False).rename(columns={'index':'case_id'})
print('train rows:', len(train_df))
print('validation rows:', len(val_df))
print('test rows:', len(test_df))
print(test_df['intent'].value_counts().sort_index())

train rows: 70
validation rows: 15
test rows: 15
intent
account_login     4
order_delivery    4
product_info      4
refund_return     3
Name: count, dtype: int64


In [3]:
def evaluate(name, y_true, y_pred, runtime, note):
    labels = sorted(set(y_true))
    return {
        'method': name,
        'accuracy': accuracy_score(y_true, y_pred),
        'macro_f1': f1_score(y_true, y_pred, average='macro'),
        'refund_return_recall': recall_score(y_true, y_pred, labels=labels, average=None)[labels.index('refund_return')],
        'threshold': 'N/A',
        'runtime_seconds': runtime,
        'note': note,
    }

rows = []
pred_frames = []

start = time.perf_counter()
majority = DummyClassifier(strategy='most_frequent')
majority.fit(train_df[['message']], train_df['intent'])
majority_pred = majority.predict(test_df[['message']])
rows.append(evaluate('Majority Class', test_df['intent'], majority_pred, time.perf_counter() - start, 'Naive label-distribution baseline'))
pred_frames.append(('Majority Class', majority_pred))

start = time.perf_counter()
rule_pred = predict_batch(test_df['message'].tolist())
rows.append(evaluate('Rule-based Keyword', test_df['intent'], rule_pred, time.perf_counter() - start, 'Week 4 credible baseline'))
pred_frames.append(('Rule-based Keyword', rule_pred))

start = time.perf_counter()
model = Pipeline([('tfidf', TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4), min_df=1)), ('clf', LogisticRegression(max_iter=1000, random_state=42))])
model.fit(train_df['message'], train_df['intent'])
ml_pred = model.predict(test_df['message'])
rows.append(evaluate('TF-IDF Logistic Regression', test_df['intent'], ml_pred, time.perf_counter() - start, 'TF-IDF fit on train only'))
pred_frames.append(('TF-IDF Logistic Regression', ml_pred))

results = pd.DataFrame(rows)
print(results.to_string(index=False))

                    method  accuracy  macro_f1  refund_return_recall threshold  runtime_seconds                              note
            Majority Class  0.266667  0.105263              0.000000       N/A         0.002463 Naive label-distribution baseline
        Rule-based Keyword  1.000000  1.000000              1.000000       N/A         0.000635          Week 4 credible baseline
TF-IDF Logistic Regression  0.733333  0.698413              0.333333       N/A         0.037751          TF-IDF fit on train only


In [4]:
all_predictions = []
for method, pred in pred_frames:
    temp = test_df[['case_id', 'message', 'intent']].copy()
    temp['method'] = method
    temp['prediction'] = pred
    temp['is_correct'] = temp['intent'] == temp['prediction']
    all_predictions.append(temp)
prediction_df = pd.concat(all_predictions, ignore_index=True)
failures = prediction_df[~prediction_df['is_correct']].copy()
priority = {'TF-IDF Logistic Regression': 0, 'Majority Class': 1, 'Rule-based Keyword': 2}
failures['priority'] = failures['method'].map(priority).fillna(9)
top3 = failures.sort_values(['priority', 'case_id']).head(3)
print(top3[['case_id', 'message', 'intent', 'prediction', 'method']].to_string(index=False))

 case_id         message         intent    prediction                     method
      14 我想查訂單編號A1023的進度 order_delivery refund_return TF-IDF Logistic Regression
      18      海外訂單大概多久到貨 order_delivery refund_return TF-IDF Logistic Regression
      28      請問七天鑑賞期怎麼算  refund_return  product_info TF-IDF Logistic Regression
